# Synthetic QA Generation 

In [1]:
!pip install -q -U transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import os

MINI_CSV   = Path('/kaggle/input/datasets/mohamedhassan77/mimic-cxr-mini/mini_dataset.csv')
QA_CSV     = Path('/kaggle/working/qa_dataset.csv')
REJECT_LOG = Path('/kaggle/working/_rejected.jsonl')
MODEL_ID   = 'google/medgemma-1.5-4b-it'

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [3]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
).eval()
print('Loaded:', MODEL_ID)

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Loaded: google/medgemma-1.5-4b-it


In [4]:
def chat_text(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': prompt}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0, inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

# Smoke test — output should contain [{...}] somewhere (thinking text may surround it)
print(chat_text('Return ONLY this JSON list: [{"a": 1}, {"b": 2}]'))

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<unused94>thought
The user wants a JSON list containing two objects.
Each object should have one key-value pair.
The first object should have the key "a" with the value 1.
The second object should have the key "b" with the value 2.

```json
[
  {"a": 1},
  {"b": 2}
]
```
This matches the requested format.<unused95>```json
[
  {"a": 1},
  {"b": 2}
]
```


In [5]:
import json, re
import pandas as pd
from tqdm.auto import tqdm

PROMPT = """Convert this radiology report into 2-3 chest X-ray VQA pairs.
Output ONLY the JSON list below. No analysis, no preamble, no thinking text.

Each pair: {{"type": one of presence/location/severity/comparison/normality,
            "question": image-grounded question (do NOT mention "the report"),
            "answer":  1-sentence clinical statement}}

Example:
Report: "Small left pleural effusion. No pneumothorax."
JSON: [{{"type":"presence","question":"Is there a pleural effusion?","answer":"Yes, a small left pleural effusion is seen."}},{{"type":"presence","question":"Is there a pneumothorax?","answer":"No pneumothorax is identified."}}]

Report: \"\"\"{report}\"\"\"
JSON:"""

ALLOWED = {'presence', 'location', 'severity', 'comparison', 'normality'}
LEAK = ('the report', 'according to the report', 'report states', 'report mentions')

def generate_triplet(report_text):
    raw = chat_text(PROMPT.format(report=report_text), max_new_tokens=1024)
    m = re.search(r'\[\s*\{.*\}\s*\]', raw, flags=re.DOTALL)
    if not m:
        return None, raw
    try:
        parsed = json.loads(m.group(0))
        return (parsed if isinstance(parsed, list) else None), raw
    except json.JSONDecodeError:
        return None, raw

def is_clean(p):
    q, a, t = str(p.get('question','')).strip(), str(p.get('answer','')).strip(), str(p.get('type','')).strip().lower()
    if t not in ALLOWED: return False
    if len(q.split()) < 3 or len(a.split()) < 3: return False
    if any(s in a.lower() for s in LEAK): return False
    return True

In [6]:
df = pd.read_csv(MINI_CSV)
done = set(pd.read_csv(QA_CSV)['study_id'].unique()) if QA_CSV.exists() else set()
todo = df[~df['study_id'].isin(done)].reset_index(drop=True).head(500)
print(f'{len(done)} already done. Processing {len(todo)} new reports.')

def append_rows(rows):
    out = pd.DataFrame(rows)
    out.to_csv(QA_CSV, mode='a', header=not QA_CSV.exists(), index=False)

def log_reject(sid, raw):
    with open(REJECT_LOG, 'a', encoding='utf-8') as f:
        f.write(json.dumps({'study_id': str(sid), 'raw': raw}) + '\n')

kept = rejected = 0
for _, row in tqdm(todo.iterrows(), total=len(todo)):
    triplet, raw = generate_triplet(row['cleaned_report'])
    if not triplet:
        log_reject(row['study_id'], raw); rejected += 1; continue
    clean = [p for p in triplet if is_clean(p)]
    if not clean:
        log_reject(row['study_id'], raw); rejected += 1; continue
    append_rows([{**p,
                  'subject_id': row['subject_id'],
                  'study_id':   row['study_id'],
                  'image_path': row['image_path']} for p in clean])
    kept += len(clean)

print(f'Done. Kept {kept} pairs. Rejected {rejected}.')

0 already done. Processing 500 new reports.


  0%|          | 0/500 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[tra

Done. Kept 1497 pairs. Rejected 153.


In [7]:
qa = pd.read_csv(QA_CSV)
print('Total:', len(qa), '| studies:', qa['study_id'].nunique())
print(qa['type'].value_counts())
qa.head(8)

Total: 1497 | studies: 347
type
presence      811
normality     443
location      131
comparison     89
severity       23
Name: count, dtype: int64


,type,question,answer,subject_id,study_id,image_path
0,location,Where does the right chest port catheter termi...,The right chest port catheter terminates in th...,16584374,s56205584,official_data_iccv_final/files/p16/p16584374/s...
1,presence,Is there an opacity at the right lung base?,"Yes, there is an opacity at the right lung bas...",16584374,s56205584,official_data_iccv_final/files/p16/p16584374/s...
2,presence,Is there a pneumothorax?,No pneumothorax is identified.,16584374,s56205584,official_data_iccv_final/files/p16/p16584374/s...
3,normality,Are the lungs clear?,The lungs are clear bilaterally.,17202146,s52566369,official_data_iccv_final/files/p17/p17202146/s...
4,presence,Is there evidence of pneumonia?,No evidence of pneumonia is seen.,17202146,s52566369,official_data_iccv_final/files/p17/p17202146/s...
5,presence,Is there pleural effusion?,No evidence of pleural effusion is present.,17202146,s52566369,official_data_iccv_final/files/p17/p17202146/s...
6,presence,Are surgical clips visible on the chest X-ray?,"Yes, surgical clips are noted overlying the th...",15952632,s50198184,official_data_iccv_final/files/p15/p15952632/s...
7,presence,"Is there lobar consolidation, large pleural ef...","No lobar consolidation, large pleural effusion...",15952632,s50198184,official_data_iccv_final/files/p15/p15952632/s...
